<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={


/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

... (87 outputs left out)

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace fo

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:604: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace 

Run date time (IST): 2026-11-03 14:31:24
=================== PORTFOLIO QUALITY ====================
qualified stocks: 86
with latest results: 86
still star stocks: 46
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.48 C
CAGR/XIRR %:     -0.23%
Est FTT Value:   2.74 C
Future Upside:   1.26 C (85.09%)
=================== PERFORMANCE ==========================
Current PnL:             -48.36 L (-25.2%)
Total Booked PnL:        48.59 L (32.6%)
Booked + Current PnL:    22.45 K (0.15%)
CY Booked PnL:           5.71 L (3.86%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               1.17 L (0.8%)
CY Investment:            1.92 C
Reserve:                  1.90 K
=================== P&L DETAIL ============================
Total Profit:             1.72 L
Total Loss:               -50.08 L


/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
85,ZYDUSLIFE,0.04,30.87,3.55,35.51,9736.0,64687.0,274261.0,1285.01,29.52,69.0,H-LC,11.55,120.0,6.64,1.85,44.33,AR,ATH,PHARMA
12,BOSCH-HCIL,1.35,-25.08,33.47,-0.01,32047.0,-32056.0,95749.0,2282.00,747.20,45.0,M-SC,21.80,231.0,-1.00,0.65,62.98,OX40N,BTT,AC
76,TTKPRESTIG,-0.91,-32.39,48.05,0.09,32738.0,-32644.0,68133.0,770.00,-17.62,47.0,M-SC,5.07,250.0,-1.00,0.46,22.63,OX40N,NTT,DURABLES
40,ICICIPRULI,-0.73,-32.65,28.34,-13.56,37602.0,-64323.0,132682.0,522.35,-56.81,24.0,X-MC,8.73,78.0,-1.71,0.90,0.00,X40,ATH,INSURANCE
66,SIS,0.30,-12.66,38.84,21.26,37665.0,-14056.0,96976.0,528.00,86.48,42.0,H-SC,8.39,168.0,-0.37,0.65,44.19,OX40N,NTT,MISC


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
52,MARUTI,1.18,-13.71,44.96,25.08,116815.0,-41298.0,259820.0,17120.06,-19.09,55.0,X-LC,14.50,33.0,-0.35,1.75,5.73,XR,ATH,AUTO
38,HYUNDAI,1.59,4.31,37.99,43.94,99594.0,10826.0,262158.0,2782.73,8.06,56.0,X-LC,6.60,29.0,0.11,1.77,20.37,X40N,ATH,AUTO
51,M&M,0.49,-9.99,30.53,17.48,82731.0,-30092.0,270983.0,3762.88,-21.52,46.0,X-LC,10.20,25.0,-0.36,1.83,4.78,X40N,ATH,AUTO
35,HDFCLIFE,-0.16,-9.68,55.62,40.56,156312.0,-30124.0,281035.0,811.41,-34.30,42.0,X-LC,10.31,31.0,-0.19,1.90,1.52,X40,ATH,INSURANCE
23,DMART,3.04,-18.25,61.99,32.43,179499.0,-64638.0,289562.0,5391.45,-31.96,45.0,X-LC,15.83,14.0,-0.36,1.95,6.20,X40N,ATH,FMCG


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TRIDENT,5.72,-37.13,104.43,28.51,60565.0,-34258.0,57996.0,48.00,-17.13,63.0,M-SC,9.19,223.0,-0.57,0.39,14.59,XR,NTT,TEXTILES
29,GREENPANEL,5.68,-64.93,308.15,43.14,197851.0,-118872.0,64206.0,537.00,NaN,37.0,NaN,24.59,225.0,-0.60,0.43,5.68,XY24,NTT,MISC
49,KANSAINER,4.29,-37.39,81.25,13.47,137175.0,-100836.0,168831.0,340.00,-34.39,56.0,H-SC,8.64,162.0,-0.74,1.14,19.33,XY24,NTT,PAINTS
15,CAMS,3.59,-9.70,37.85,24.48,86078.0,-24436.0,227420.0,950.00,-13.96,51.0,X-SC,6.64,89.0,-0.28,1.53,11.64,X40N,NTT,MISC
58,RAJOOENG,3.41,-46.22,158.93,39.24,87602.0,-47380.0,55120.0,142.72,NaN,50.0,NaN,4.99,136.0,-0.54,0.37,18.41,AR,ATH,MISC


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
64,SFL,-6.91,-49.62,151.37,26.63,199954.0,-130122.0,132096.0,1287.0,NaN,22.0,NaN,6.27,242.0,-0.65,0.89,10.25,XY24,NTT,MISC
80,VALIANTORG,-4.14,-14.17,102.91,74.15,144506.0,-23185.0,140420.0,838.0,NaN,46.0,NaN,49.41,145.0,-0.16,0.95,108.97,XR,NTT,CHEMICALS
24,EASEMYTRIP,-2.44,-66.89,340.00,45.70,199063.0,-118267.0,58548.0,26.4,NaN,57.0,NaN,17.78,204.0,-0.59,0.40,12.57,XY24,NTT,TRAVEL
73,TITAGARH,-1.67,-35.60,126.30,45.74,196981.0,-86214.0,155963.0,1548.0,NaN,38.0,NaN,8.63,175.0,-0.44,1.05,19.14,XY24,NTT,ENGINEERING
71,TATAELXSI,-1.67,-46.38,146.22,32.03,103364.0,-61144.0,70691.0,9161.0,-39.11,59.0,H-SC,14.31,158.0,-0.59,0.48,24.19,OX40N,NTT,IT


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
67,STARHEALTH,0.68,-0.25,47.62,47.26,133542.0,-691.0,280432.0,761.0,-12.12,38.0,H-SC,5.34,173.0,-0.01,1.89,21.25,XY24,NTT,INSURANCE


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,0.42,44.67,42.70,106.44,91203.0,65947.0,213591.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.44,61.84,OX40N,ATH,CABLES
66,SIS,0.30,-12.66,38.84,21.26,37665.0,-14056.0,96976.0,528.00,86.48,42.0,H-SC,8.39,168.0,-0.37,0.65,44.19,OX40N,NTT,MISC
12,BOSCH-HCIL,1.35,-25.08,33.47,-0.01,32047.0,-32056.0,95749.0,2282.00,747.20,45.0,M-SC,21.80,231.0,-1.00,0.65,62.98,OX40N,BTT,AC
76,TTKPRESTIG,-0.91,-32.39,48.05,0.09,32738.0,-32644.0,68133.0,770.00,-17.62,47.0,M-SC,5.07,250.0,-1.00,0.46,22.63,OX40N,NTT,DURABLES
49,KANSAINER,4.29,-37.39,81.25,13.47,137175.0,-100836.0,168831.0,340.00,-34.39,56.0,H-SC,8.64,162.0,-0.74,1.14,19.33,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,0.42,44.67,42.70,106.44,91203.0,65947.0,213591.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.44,61.84,OX40N,ATH,CABLES
85,ZYDUSLIFE,0.04,30.87,3.55,35.51,9736.0,64687.0,274261.0,1285.01,29.52,69.0,H-LC,11.55,120.0,6.64,1.85,44.33,AR,ATH,PHARMA


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,0.42,44.67,42.7,106.44,91203.0,65947.0,213591.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.44,61.84,OX40N,ATH,CABLES


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,0.42,44.67,42.70,106.44,91203.0,65947.0,213591.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.44,61.84,OX40N,ATH,CABLES
84,WIPRO,0.50,-9.29,112.79,93.01,256019.0,-23253.0,226987.0,420.00,-23.61,71.0,M-LC,8.84,98.0,-0.09,1.53,25.89,XR,NTT,IT
80,VALIANTORG,-4.14,-14.17,102.91,74.15,144506.0,-23185.0,140420.0,838.00,NaN,46.0,NaN,49.41,145.0,-0.16,0.95,108.97,XR,NTT,CHEMICALS
70,TANLA,2.56,-25.18,187.60,115.17,380025.0,-68189.0,202572.0,1903.93,NaN,79.0,NaN,9.71,147.0,-0.18,1.37,79.07,AR,ATH,IT
42,INDIAMART,2.42,-33.90,203.97,100.92,166287.0,-41811.0,81525.0,4675.60,NaN,41.0,NaN,16.85,140.0,-0.25,0.55,3.43,AR,ATH,MISC


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
37,HONAUT,-0.34,-24.52,99.91,50.88,145419.0,-47288.0,145550.0,58192.79,-41.02,22.0,X-SC,5.93,90.0,-0.33,0.98,10.89,X40N,ATH,ELECTRICAL
40,ICICIPRULI,-0.73,-32.65,28.34,-13.56,37602.0,-64323.0,132682.0,522.35,-56.81,24.0,X-MC,8.73,78.0,-1.71,0.90,0.00,X40,ATH,INSURANCE
6,AWL,-0.65,-43.29,194.49,67.02,393016.0,-154234.0,202075.0,485.00,-58.24,32.0,X-SC,6.38,81.0,-0.39,1.36,0.46,XY24,NTT,FMCG
7,BAJAJHFL,-0.04,-30.19,132.25,62.13,252079.0,-82438.0,190608.0,181.50,-41.76,39.0,X-MC,5.09,60.0,-0.33,1.29,6.92,X40N,ATH,FINANCE
8,BAJAJHLDNG,0.11,-7.20,36.84,26.99,106979.0,-22539.0,290388.0,14191.91,-15.86,40.0,X-LC,4.15,28.0,-0.21,1.96,20.75,X40,ATH,FINANCE


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
55,PAGEIND,-0.93,-8.27,44.83,32.85,110367.0,-22189.0,246190.0,50936.07,-31.74,47.0,X-MC,1.34,55.0,-0.20,1.66,17.56,X40,ATH,APPARELS
1,ABBOTINDIA,-0.31,-13.65,33.61,15.37,51907.0,-24415.0,154440.0,34390.26,-29.65,47.0,X-MC,1.65,58.0,-0.47,1.04,4.62,X40,ATH,PHARMA
17,CIPLA,-0.33,3.91,19.22,23.89,60048.0,11767.0,312422.0,1648.14,17.84,59.0,X-LC,2.23,18.0,0.20,2.11,17.02,X40N,ATH,PHARMA
11,BERGEPAINT,1.98,-11.93,42.29,25.32,95999.0,-30747.0,227002.0,680.00,-23.45,57.0,X-MC,2.36,68.0,-0.32,1.53,19.04,XY24,NTT,PAINTS
33,HCLTECH,-0.12,-1.99,28.26,25.70,99978.0,-7195.0,353778.0,1837.07,-6.73,71.0,X-LC,3.95,8.0,-0.07,2.39,41.15,X40,ATH,IT


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
40,ICICIPRULI,-0.73,-32.65,28.34,-13.56,37602.0,-64323.0,132682.0,522.35,-56.81,24.0,X-MC,8.73,78.0,-1.71,0.90,0.00,X40,ATH,INSURANCE
6,AWL,-0.65,-43.29,194.49,67.02,393016.0,-154234.0,202075.0,485.00,-58.24,32.0,X-SC,6.38,81.0,-0.39,1.36,0.46,XY24,NTT,FMCG
28,GODREJCP,0.67,-24.77,76.66,32.90,143913.0,-61808.0,187729.0,1460.97,-26.23,40.0,X-MC,16.88,32.0,-0.43,1.27,1.10,X40N,ATH,FMCG
35,HDFCLIFE,-0.16,-9.68,55.62,40.56,156312.0,-30124.0,281035.0,811.41,-34.30,42.0,X-LC,10.31,31.0,-0.19,1.90,1.52,X40,ATH,INSURANCE
19,COLPAL,3.09,-38.35,123.44,37.76,200430.0,-100995.0,162370.0,3628.04,-25.25,41.0,X-MC,14.11,62.0,-0.50,1.10,3.26,XY25,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
56,PGHH,2.67,-47.03,147.36,31.02,166658.0,-100424.0,113096.0,17484.41,-61.09,52.0,X-SC,24.91,57.0,-0.60,0.76,6.47,X40,ATH,FMCG
40,ICICIPRULI,-0.73,-32.65,28.34,-13.56,37602.0,-64323.0,132682.0,522.35,-56.81,24.0,X-MC,8.73,78.0,-1.71,0.90,0.00,X40,ATH,INSURANCE
2,ACC,2.03,-47.41,232.43,74.81,327726.0,-127128.0,141000.0,3906.00,-47.65,50.0,X-SC,16.47,85.0,-0.39,0.95,4.45,XY24,BTT,CEMENT
37,HONAUT,-0.34,-24.52,99.91,50.88,145419.0,-47288.0,145550.0,58192.79,-41.02,22.0,X-SC,5.93,90.0,-0.33,0.98,10.89,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,-0.31,-13.65,33.61,15.37,51907.0,-24415.0,154440.0,34390.26,-29.65,47.0,X-MC,1.65,58.0,-0.47,1.04,4.62,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,ITC,0.73,-23.59,68.44,28.70,228839.0,-103231.0,334364.0,452.00,-31.16,57.0,X-LC,8.60,1.0,-0.45,2.26,5.24,X40,NTT,FMCG
72,TCS,-0.47,-25.01,69.01,26.75,252597.0,-122045.0,366030.0,4208.34,-39.28,66.0,X-LC,5.20,2.0,-0.48,2.47,26.95,X40,ATH,IT
45,INFY,0.24,-17.69,73.33,42.66,241940.0,-70922.0,329933.0,1972.00,-40.47,60.0,X-LC,9.19,3.0,-0.29,2.23,18.11,X40,NTT,IT
74,TMPV,1.05,-33.00,117.98,46.05,353642.0,-147625.0,299747.0,600.00,-87.07,46.0,X-LC,17.83,4.0,-0.42,2.02,5.38,XY24,NTT,AUTO
33,HCLTECH,-0.12,-1.99,28.26,25.70,99978.0,-7195.0,353778.0,1837.07,-6.73,71.0,X-LC,3.95,8.0,-0.07,2.39,41.15,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,1.35,-25.08,33.47,-0.01,32047.0,-32056.0,95749.0,2282.00,747.20,45.0,M-SC,21.80,231.0,-1.00,0.65,62.98,OX40N,BTT,AC
80,VALIANTORG,-4.14,-14.17,102.91,74.15,144506.0,-23185.0,140420.0,838.00,NaN,46.0,NaN,49.41,145.0,-0.16,0.95,108.97,XR,NTT,CHEMICALS
70,TANLA,2.56,-25.18,187.60,115.17,380025.0,-68189.0,202572.0,1903.93,NaN,79.0,NaN,9.71,147.0,-0.18,1.37,79.07,AR,ATH,IT
25,FINCABLES,0.42,44.67,42.70,106.44,91203.0,65947.0,213591.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.44,61.84,OX40N,ATH,CABLES
85,ZYDUSLIFE,0.04,30.87,3.55,35.51,9736.0,64687.0,274261.0,1285.01,29.52,69.0,H-LC,11.55,120.0,6.64,1.85,44.33,AR,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
70,TANLA,2.56,-25.18,187.60,115.17,380025.0,-68189.0,202572.0,1903.93,NaN,79.0,NaN,9.71,147.0,-0.18,1.37,79.07,AR,ATH,IT
84,WIPRO,0.50,-9.29,112.79,93.01,256019.0,-23253.0,226987.0,420.00,-23.61,71.0,M-LC,8.84,98.0,-0.09,1.53,25.89,XR,NTT,IT
33,HCLTECH,-0.12,-1.99,28.26,25.70,99978.0,-7195.0,353778.0,1837.07,-6.73,71.0,X-LC,3.95,8.0,-0.07,2.39,41.15,X40,ATH,IT
71,TATAELXSI,-1.67,-46.38,146.22,32.03,103364.0,-61144.0,70691.0,9161.00,-39.11,59.0,H-SC,14.31,158.0,-0.59,0.48,24.19,OX40N,NTT,IT
72,TCS,-0.47,-25.01,69.01,26.75,252597.0,-122045.0,366030.0,4208.34,-39.28,66.0,X-LC,5.20,2.0,-0.48,2.47,26.95,X40,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.19
1,25,49.42
2,50,80.35


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    42.72
MC    20.59
SC    20.00
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.57
XY24     22.21
X40N     18.46
XR        9.83
XY25      9.48
OX40N     6.56
AR        5.13
SR        0.78
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    39.34
X-MC    18.23
H-SC     7.95
X-SC     7.05
M-SC     5.00
H-LC     1.85
M-LC     1.53
M-MC     1.05
H-MC     0.98
L-MC     0.33
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,17.74,-30.49,77.44
IT,13.23,-30.99,99.46
FINANCE,10.42,-30.53,86.29
AUTO,8.26,-22.37,66.02
PHARMA,6.39,6.19,19.67
INSURANCE,5.82,-14.87,46.11
MISC,5.42,-53.28,114.89
PAINTS,5.36,-30.10,53.05
BANKS,4.91,-23.14,84.89


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4141617.0,19
X40,2309026.0,17
X40N,1604996.0,12
XR,1598937.0,13
AR,1034454.0,7
OX40N,817772.0,10
XY25,809539.0,6
SR,297274.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,3113069.0,19
X-MC,1710515.0,14
X-SC,1227790.0,6
H-SC,970929.0,8
M-SC,423068.0,6
H-MC,267787.0,1
M-LC,256019.0,1
M-MC,89314.0,1
L-MC,54074.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1366983.0      8
X-SC       XY24       720742.0      2
X-LC       X40N       633047.0      5
           XY24       598722.0      2
H-SC       XY24       596257.0      3
X-MC       X40N       572688.0      4
           X40        546290.0      6
X-LC       XY25       397502.0      3
X-MC       XY25       322723.0      2
X-SC       X40        275551.0      2
X-MC       XY24       268814.0      2
H-MC       XY24       267787.0      1
M-LC       XR         256019.0      1
H-SC       OX40N      252331.0      3
X-SC       X40N       231497.0      2
M-SC       OX40N      222980.0      4
           XY24       139523.0      1
H-SC       XR         122341.0      2
X-LC       XR         116815.0      1
M-MC       XY25        89314.0      1
M-SC       XR          60565.0      1
L-MC       XR          54074.0      1
H-LC       AR           9736.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 27.0 seconds
